<h1> Genetic Programming (GP) </h1>

Philosophically, this algorithm differs fundamentally from GA and DE.

So far:

GA → Finding the best solution</br>
DE → Finding the best parameter values

But GP:

It attempts to generate a program, formula, or computational structure itself and Our answer is no longer a number or a vector, but rather a program tree, it meansWe find the program or function itself that solves the problem.

### Difference between GA and GP

|                | GA                 | GP                |
| -------------- | ------------------ | ----------------- |
| Goal           | Finding the Solution | Finding the Program |
| Representation | Chromosome         | Tree              |
| Gene           | Value              | Function/Operator |
| Crossover      | Gene segment exchange | Swapping Subtrees   |
| Mutation       | Changing Gene         | Changing Node/Branch |


As previously mentioned, the main difference between GP and GA/DE lies in the representation.

In GA:</br>
Solution → Chromosome → Array

In DE:</br>
Solution → Vector → [x1, x2, ...]

However, in GP:</br>
Solution → Program → Tree

Thus, our first step is to construct an expression tree.

In [1]:
import numpy as np

from src.GP import (
    Node,
    generate_random_tree,
    fitness_function,
    tournament_selection,
    subtree_crossover,
    subtree_mutation,
    genetic_programming,
    NodeAdvanced,
    generate_random_tree_advanced,
    subtree_crossover_advanced,
    subtree_mutation_advanced,
    genetic_programming_advanced
)
from problems.symbolic_regression import create_dataset

<h3> Step 1: Tree representation & Evaluation </h3>

In [2]:
tree = Node(
    "*",
    Node(
        "+",
        Node("x"),
        Node(1)
    ),
    Node("x")
)

print(tree.to_string())

((x + 1) * x)


In [3]:
tree.evaluate(3)

12

<h3> Step 2: Random Tree Generator </h3>

So far, we have managed to construct a tree manually.

However, a true GP system must be able to:

generate a large number of random programs,</br>
place them into a population,</br>
and then initiate the evolution process.

Therefore, we need a function that generates a random expression tree.

In [4]:
tree = generate_random_tree(
    max_depth=3
)

print(tree.to_string())

2


In [5]:
for i in range(5):
    tree = generate_random_tree(
        max_depth=3
    )
    print(
        tree.to_string()
    )

2
2
(((3 / x) + (x - 1)) / (3 / (x + x)))
(2 - ((x * x) / (3 - 1)))
(x * ((x + 1) / (2 - 3)))


<h3> Step 3: Symbolic Regression Function </h3>

In [6]:
x_data, y_data = create_dataset(start=-5, end=5, samples=50)

print(x_data[:5])
print(y_data[:5])

[-5.         -4.79591837 -4.59183673 -4.3877551  -4.18367347]
[16.         14.40899625 12.90129113 11.47688463 10.13577676]


<h3> Step 4: Fitness Function </h3>

For the entire dataset:

We use:

Mean Squared Error (MSE)

$MSE= \frac1n \sum(y-\hat y)^2$

$x∗x+(2∗x+1)$

In [7]:
tree = Node(
    "+",
    Node(
        "*",
        Node("x"),
        Node("x")
    ),
    Node(
        "+",
        Node(
            "*",
            Node(2),
            Node("x")
        ),
        Node(1)
    )
)

In [8]:
fitness_function(
    tree,
    x_data,
    y_data,
)

0.0

<h3> Step 5: GP Selection </h3>

In GP, ​​just as in GA, the selection process is responsible for:

Selecting better programs with higher probability.</br>
Giving weaker programs a lower chance of selection.

In [9]:
population = []

for _ in range(5):
    tree = generate_random_tree(
        max_depth=3
    )
    population.append(tree)

In [10]:
fitness_values = []

for tree in population:
    fitness_values.append(
        fitness_function(
            tree,
            x_data,
            y_data,
        )
    )

In [11]:
for i in range(len(population)):
    print(
        population[i].to_string(),
        "->",
        fitness_values[i]
    )

(((1 * x) - 3) + ((3 / 3) * x)) -> 220.72783449073094
((3 / (1 / 2)) * ((2 - 1) - (2 - x))) -> 444.5441610213431
(1 + ((2 / 2) * (2 * x))) -> 135.34007938869013
(2 - x) -> 197.05436510297585
3 -> 139.34007938869013


In [12]:
parent = tournament_selection(
    population,
    fitness_values,
)

print(parent.to_string())

(1 + ((2 / 2) * (2 * x)))


<h3> Step 6: Subtree Crossover </h3>

In GP, ​​crossover is typically the primary operator.

This contrasts with GA, where:</br>
Crossover</br>
Mutation</br>
are of roughly equal importance.

In GP:</br>
Typically:</br>
Crossover ≈ 80–90%</br>
Mutation ≈ 10–20%

This is because combining good programs usually yields better results than randomly altering the structure.

In [13]:
parent1 = Node(
    "*",
    Node(
        "+",
        Node("x"),
        Node(1)
    ),
    Node("x")
)

parent2 = Node(
    "-",
    Node("x"),
    Node(5)
)

print(parent1.to_string())
print(parent2.to_string())

((x + 1) * x)
(x - 5)


In [14]:
child = subtree_crossover(
    parent1,
    parent2
)

print(child.to_string())

((x + x) * x)


<h3> Step 7: Tree Mutation </h3>

Unlike in GA:

Mutation in GP means:

Changing an operator</br>
Replacing a subtree</br>
Changing a terminal

#### There are several types of mutation:

1) Node Mutation: Only the value of a node changes.

2) Terminal Mutation: The value of a terminal changes.

3) Subtree Mutation (the most important): An entire subtree is removed and replaced by a new, randomly generated tree. This type of mutation leads to increased diversity.

Mutation is much more powerful in GP than in GA,

because a small change in the tree can create a completely new structure.

In [15]:
tree = Node(
    "*",
    Node("x"),
    Node(2)
)

print(tree.to_string())

(x * 2)


In [16]:
mutated = subtree_mutation(
    tree,
    max_depth=2
)

print(mutated.to_string())

(1 * 2)


<h3> Step 8: Complete Genetic Programming Algorithm </h3>

Initialize Population
        |
        ↓
Evaluate Fitness
        |
        ↓
Select Parents
        |
        ↓
Create New Individuals
        |
        ├── Crossover
        |
        ├── Mutation
        |
        ↓
Replace Population
        |
        ↓
Repeat

In [17]:
x_data, y_data = create_dataset(start=-10, end=10, samples=100)

result = genetic_programming(
    x_data,
    y_data,
    population_size=100,
    generations=15,
    max_depth=4,
)

In [18]:
print("Best Tree:")
print(result["best_tree"].to_string())

print("\nFitness:")
print(result["fitness"])

print("\nHistory:")
print(result["history"])

Best Tree:
((3 * x) + (((2 / (x + (((2 / (2 / 1)) + ((x * x) - 2)) + ((3 * (2 - 3)) / ((2 / (2 / 1)) + ((x * x) - 2)))))) + ((x * x) - (x - 3))) + (1 - 3)))

Fitness:
0.7736930244276568

History:
[59.006734006734014, 35.53180244199683, 34.06923400673401, 34.06923400673401, 34.06923400673401, 34.06923400673401, 34.06923400673401, 34.06923400673401, 1.2669336885455171, 1.2669336885455171, 1.2669336885455171, 1.2669336885455171, 1.2669336885455171, 1.2669336885455171, 0.7736930244276568]


#### GP Symbolic Regression Result

The Genetic Programming algorithm was evaluated on a symbolic regression problem where the target function was:
$y=x^2+2x+1$

The obtained expression was:
$((3+(x-3))+2)x$

which can be simplified to:
$x(x+2)=x^2+2x$

The result successfully discovered the main structure of the target function, but it missed the constant term (+1), resulting in:

$MSE \approx 1$

The convergence curve shows a rapid improvement in early generations followed by stagnation, indicating premature convergence around a near-optimal solution.

This behavior highlights the importance of maintaining diversity in Genetic Programming to explore alternative program structures.

<h3> Step 9: Advanced GP + Harder Benchmark </h3>

The implementation was progressively extended from a simple binary-tree GP representation to a more flexible expression-tree representation supporting arbitrary arity functions and protected operators.

In [19]:
tree = NodeAdvanced(
    "sin",
    [
        NodeAdvanced("x")
    ]
)

print(
    tree.evaluate(
        np.pi/2
    )
)

1.0


In [20]:
tree = NodeAdvanced(
    "*",
    [
        NodeAdvanced(
            "+",
            [
                NodeAdvanced("x"),
                NodeAdvanced(2)
            ]
        ),
        NodeAdvanced("x")
    ]
)

tree.evaluate(3)

15

In [21]:
tree = NodeAdvanced(
    "sin",
    [
        NodeAdvanced("x")
    ]
)

print(tree.to_string())

sin(x)


In [22]:
tree = NodeAdvanced(
    "+",
    [
        NodeAdvanced(
            "sin",
            [
                NodeAdvanced("x")
            ]
        ),
        NodeAdvanced(
            "*",
            [
                NodeAdvanced("x"),
                NodeAdvanced("x")
            ]
        )
    ]
)

print(tree.to_string())

(sin(x) + (x * x))


In [23]:
nodes = tree.get_nodes()

print(len(nodes))

6


In [24]:
tree = generate_random_tree_advanced(
    max_depth=3
)

print(tree.to_string())

((exp(x) + x) + (sin(x) + log(-4.73)))


In [25]:
for i in range(5):
    tree = generate_random_tree_advanced(
        max_depth=3
    )
    print(tree.to_string())

-0.9
exp(((0.59 / x) + (-2.2 * x)))
exp((log(x) / cos(x)))
x
log(cos(sin(-0.88)))


### Advanced Fitness Function

In real-world GP, we face two problems:

Some expressions may be invalid.</br>
Some expressions become excessively large (the "bloat" problem).

1) Safe Evaluation 
2) Bloat  </br>
$Fitness = MSE+\lambda Size(Tree)$

This is what is known in GP as:</br>
**Parsimony Pressure**

Meaning:</br>
Between two programs with similar performance, the simpler program is preferred.


### Advanced Crossover

In [26]:
parent1 = generate_random_tree_advanced(3)
parent2 = generate_random_tree_advanced(3)

print(parent1.to_string())
print(parent2.to_string())

3.21
log(exp(log(x)))


In [27]:
child = subtree_crossover_advanced(
    parent1,
    parent2
)

print(child.to_string())

x


### Advanced Mutation

In [28]:
tree = generate_random_tree_advanced(3)

print(tree.to_string())

((exp(x) - (x + x)) + 4.8)


In [29]:
mutated = subtree_mutation_advanced(tree)

print(mutated.to_string())

((exp(x) - (cos(x) / cos((x * -3.72)))) + 4.8)


### Advanced Genetic Programming Loop

In [32]:
x_data, y_data = create_dataset(start=-10, end=10, samples=100)

result = genetic_programming_advanced(
    x_data,
    y_data,
    population_size=100,
    generations=10,
    max_depth=4,
)

In [33]:
print("Best Tree:")
print(result["best_tree"].to_string())

print("\nFitness:")
print(result["fitness"])

print("\nHistory:")
print(result["history"])

Best Tree:
((x + ((x * x) + x)) + cos(cos((x / 4.03))))

Fitness:
0.07278883749031344

History:
[1196.5524969961614, 1196.5524969961614, 136.16153602693603, 1.0089759199584603, 1.0089759199584603, 1.0089759199584603, 1.0089759199584603, 1.0089759199584603, 0.09580459772797342, 0.07278883749031344]


### Advanced GP Result

The Advanced Genetic Programming model successfully discovered the main structure of the target function.

The obtained expression:
$((x+((x*x)+x))+cos(cos(x/4.03)))$

can be simplified approximately to:
$x^2+2x+C$

where the last term behaves as a constant approximation.

Compared with the basic GP implementation (MSE ≈ 1), the advanced version significantly improved the result and reduced the error to:

$Fitness \approx 0.073$

This improvement demonstrates the benefit of flexible tree representation, random constants, and extended function sets.